In [1]:
from getpass import getpass

GITHUB_TOKEN = getpass("GitHub token (input hidden): ").strip()

if not GITHUB_TOKEN:
    raise ValueError("GitHub token was not entered.")

print("GitHub token received. It is kept only in notebook memory.")

GitHub token (input hidden):  ········


GitHub token received. It is kept only in notebook memory.


In [2]:
from pathlib import Path
import base64
import json
import urllib.request

REPOSITORY = "Creptos32/comfyui-telegram-bot"
FILES_TO_DOWNLOAD = ["bot.py", "BASE_API.json", "requirements.txt"]
BOT_DIR = Path("/workspace/telegram_bot")

BOT_DIR.mkdir(parents=True, exist_ok=True)

for file_name in FILES_TO_DOWNLOAD:
    url = f"https://api.github.com/repos/{REPOSITORY}/contents/{file_name}"
    request = urllib.request.Request(
        url,
        headers={
            "Accept": "application/vnd.github+json",
            "Authorization": f"Bearer {GITHUB_TOKEN}",
        },
    )

    with urllib.request.urlopen(request) as response:
        file_data = json.load(response)

    if file_data.get("type") != "file":
        raise RuntimeError(f"GitHub did not return a file: {file_name}")

    content = base64.b64decode(file_data["content"])
    (BOT_DIR / file_name).write_bytes(content)

del GITHUB_TOKEN
print(f"Downloaded {len(FILES_TO_DOWNLOAD)} files to {BOT_DIR}")

Downloaded 3 files to /workspace/telegram_bot


In [3]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "-r",
        "/workspace/telegram_bot/requirements.txt",
    ],
    check=True,
)

print("Python libraries installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 769.4/769.4 kB 11.0 MB/s  0:00:00
Python libraries installed.


In [5]:
from getpass import getpass
from pathlib import Path
import os

telegram_token = getpass("Telegram bot token (input hidden): ").strip()

if not telegram_token:
    raise ValueError("Telegram bot token was not entered.")

env_path = Path("/workspace/telegram_bot/.env")
env_path.write_text(f"TELEGRAM_BOT_TOKEN={telegram_token}\n", encoding="utf-8")
os.chmod(env_path, 0o600)

del telegram_token
print(f"Created protected file: {env_path}")

Telegram bot token (input hidden):  ········


Created protected file: /workspace/telegram_bot/.env


In [8]:
from pathlib import Path
import subprocess
import sys

bot_dir = Path("/workspace/telegram_bot")
required_files = ["bot.py", "BASE_API.json", "requirements.txt", ".env"]

missing = [name for name in required_files if not (bot_dir / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing files: {', '.join(missing)}")

subprocess.run(
    [sys.executable, "-m", "py_compile", str(bot_dir / "bot.py")],
    check=True,
)

print("Bot files and configuration passed validation.")

Bot files and configuration passed validation.


In [9]:
from pathlib import Path
import os
import subprocess
import sys
import time
import requests

bot_dir = Path("/workspace/telegram_bot")
api_url = "http://127.0.0.1:18188/system_stats"
pid_path = bot_dir / "bot.pid"

if pid_path.exists():
    try:
        existing_pid = int(pid_path.read_text(encoding="utf-8").strip())
        os.kill(existing_pid, 0)
        raise RuntimeError(
            f"Bot is already running (PID {existing_pid}). "
            "Do not start a second copy."
        )
    except ProcessLookupError:
        pid_path.unlink()

try:
    response = requests.get(api_url, timeout=10)
    response.raise_for_status()
except requests.RequestException as error:
    raise RuntimeError(
        "ComfyUI API is not available at 127.0.0.1:18188. "
        "Start ComfyUI first, then run this cell again."
    ) from error

log_path = bot_dir / "bot.log"
with log_path.open("a", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        [sys.executable, "bot.py"],
        cwd=bot_dir,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

pid_path.write_text(str(process.pid), encoding="utf-8")
time.sleep(2)

if process.poll() is not None:
    raise RuntimeError(f"Bot stopped immediately. Check log: {log_path}")

print(f"Bot started in the background. PID: {process.pid}")
print(f"Log file: {log_path}")

RuntimeError: Bot stopped immediately. Check log: /workspace/telegram_bot/bot.log